In [ ]:
def powers_of_two_mod(base, mod, max_power):
    power = 1
    current = base % mod
    results = {}

    while power <= max_power:
        results[power] = current
        print(f"{base}**{power} mod {mod} = {current}")
        current = (current * current) % mod
        power *= 2

    return results

# Example usage:
cap = powers_of_two_mod(547, 827, 2048)


547**1 mod 827 = 547
547**2 mod 827 = 662
547**4 mod 827 = 761
547**8 mod 827 = 221
547**16 mod 827 = 48
547**32 mod 827 = 650
547**64 mod 827 = 730
547**128 mod 827 = 312
547**256 mod 827 = 585
547**512 mod 827 = 674
547**1024 mod 827 = 253
547**2048 mod 827 = 330


In [7]:
print("RC4 Algorithm")


def rc4_state(initial_s, key):
    S = list(initial_s)
    K = [key[i % len(key)] for i in range(len(S))]
    print("Initialization:")
    print(f"  S = {S}")
    print(f"  K = {K}\n")

    # KSA
    j = 0
    print("Key-Scheduling Algorithm (KSA):")
    for i in range(len(S)):
        old_j = j
        j = (old_j + S[i] + K[i]) % len(S)
        print(f" i={i:2d}: j = ({old_j} + S[{i}]={S[i]} + K[{i}]={K[i]}) mod {len(S)} = {j}")
        print(f"    → swap S[{i}]={S[i]} ↔ S[{j}]={S[j]}")
        S[i], S[j] = S[j], S[i]
        print(f"    → S = {S}\n")
    return S

def rc4_prga(S, plaintext, n_bytes):
    i = j = 0
    cipher = []
    print("Pseudo-Random Generation Algorithm (PRGA):")
    print("Swap Happens between S[i]<->S[j]")
    print(f"    → S = {S}\n")
    for idx in range(n_bytes):
        old_i, old_j = i, j
        i = (old_i + 1) % len(S)
        j = (old_j + S[i]) % len(S)
        print(f" n={idx:2d}: i = ({old_i} + 1) mod {len(S)} = {i};  j = ({old_j} + S[{i}]={S[i]}) mod {len(S)} = {j}")
        print(f"    → swap S[{i}]={S[i]} ↔ S[{j}]={S[j]}")
        S[i], S[j] = S[j], S[i]
        t = (S[i] + S[j]) % len(S)
        k = S[t]
        c = plaintext[idx] ^ k
        print(f"    → t = (S[{i}]={S[i]} + S[{j}]={S[j]}) mod {len(S)} = {t}")
        print(f"    → k = S[{t}] = {k}")
        print(f"    → pt = {plaintext[idx]}; ct = pt⊕k = {plaintext[idx]}⊕{k} = {c}\n")
        cipher.append(c)
    print(f"Ciphertext bytes: {cipher}")
    return cipher

if __name__ == "__main__":
    initial_S  = list(range(8))          # [0,1,2,3,4,5,6,7]
    key_stream  = [1,2,3,6,1,2,3,6]       # repeated key
    plaintext   = [1,2,2,2]              # four bytes

    S = rc4_state(initial_S, key_stream)
    rc4_prga(S, plaintext, len(plaintext))


RC4 Algorithm
Initialization:
  S = [0, 1, 2, 3, 4, 5, 6, 7]
  K = [1, 2, 3, 6, 1, 2, 3, 6]

Key-Scheduling Algorithm (KSA):
 i= 0: j = (0 + S[0]=0 + K[0]=1) mod 8 = 1
    → swap S[0]=0 ↔ S[1]=1
    → S = [1, 0, 2, 3, 4, 5, 6, 7]

 i= 1: j = (1 + S[1]=0 + K[1]=2) mod 8 = 3
    → swap S[1]=0 ↔ S[3]=3
    → S = [1, 3, 2, 0, 4, 5, 6, 7]

 i= 2: j = (3 + S[2]=2 + K[2]=3) mod 8 = 0
    → swap S[2]=2 ↔ S[0]=1
    → S = [2, 3, 1, 0, 4, 5, 6, 7]

 i= 3: j = (0 + S[3]=0 + K[3]=6) mod 8 = 6
    → swap S[3]=0 ↔ S[6]=6
    → S = [2, 3, 1, 6, 4, 5, 0, 7]

 i= 4: j = (6 + S[4]=4 + K[4]=1) mod 8 = 3
    → swap S[4]=4 ↔ S[3]=6
    → S = [2, 3, 1, 4, 6, 5, 0, 7]

 i= 5: j = (3 + S[5]=5 + K[5]=2) mod 8 = 2
    → swap S[5]=5 ↔ S[2]=1
    → S = [2, 3, 5, 4, 6, 1, 0, 7]

 i= 6: j = (2 + S[6]=0 + K[6]=3) mod 8 = 5
    → swap S[6]=0 ↔ S[5]=1
    → S = [2, 3, 5, 4, 6, 0, 1, 7]

 i= 7: j = (5 + S[7]=7 + K[7]=6) mod 8 = 2
    → swap S[7]=7 ↔ S[2]=5
    → S = [2, 3, 7, 4, 6, 0, 1, 5]

Pseudo-Random Generation Al

In [8]:
print("=== RC4 Stream Cipher Demo ===\n")

def rc4_state(initial_s, key):
    """
    Build and mix the state array S using the Key-Scheduling Algorithm.

    Args:
      initial_s: list of ints, initial permutation (0…N-1)
      key: list of ints, your secret key repeated/truncated to length N

    Returns:
      A scrambled state array S ready for PRGA.
    """
    S = list(initial_s)
    # Repeat or truncate key to match length of S
    K = [key[i % len(key)] for i in range(len(S))]

    print(">> Initialization")
    print(f"   Starting S (identity): {S}")
    print(f"   Expanded Key array K:  {K}\n")

    # KSA: mix S based on key bytes
    j = 0
    print(">> Key-Scheduling Algorithm (KSA) Steps")
    for i in range(len(S)):
        old_j = j
        # Compute new j with current S[i] and K[i]
        j = (old_j + S[i] + K[i]) % len(S)
        print(f"  Step i={i:2d}:")
        print(f"    j = ({old_j} + S[{i}]={S[i]} + K[{i}]={K[i]}) mod {len(S)} → {j}")
        print(f"    Swap S[{i}] ({S[i]}) with S[{j}] ({S[j]})")
        S[i], S[j] = S[j], S[i]
        print(f"    New S: {S}\n")

    return S


def rc4_prga(S, plaintext, n_bytes):
    """
    Generate keystream bytes and encrypt plaintext via XOR.

    Args:
      S: list of ints, the mixed state from KSA
      plaintext: list of ints, data bytes to encrypt
      n_bytes: int, how many bytes to process

    Returns:
      List of ciphertext byte values.
    """
    i = j = 0
    cipher = []

    print(">> Pseudo-Random Generation Algorithm (PRGA) Steps")
    print(f"   Initial S before PRGA: {S}\n")

    for idx in range(n_bytes):
        old_i, old_j = i, j

        # 1) Advance indices
        i = (old_i + 1) % len(S)
        j = (old_j + S[i]) % len(S)
        print(f"  Round {idx}:")
        print(f"    i = ({old_i} + 1) mod {len(S)} = {i}")
        print(f"    j = ({old_j} + S[{i}]={S[i]}) mod {len(S)} = {j}")

        # 2) Swap to keep S evolving
        print(f"    Swap S[{i}] ({S[i]}) with S[{j}] ({S[j]})")
        S[i], S[j] = S[j], S[i]

        # 3) Select keystream byte
        t = (S[i] + S[j]) % len(S)
        k = S[t]
        print(f"    t = (S[{i}]={S[i]} + S[{j}]={S[j]}) mod {len(S)} = {t}")
        print(f"    Keystream byte k = S[{t}] = {k}")

        # 4) XOR with plaintext
        pt = plaintext[idx]
        ct = pt ^ k
        print(f"    Plaintext byte pt = {pt}")
        print(f"    Ciphertext byte ct = pt⊕k = {pt}⊕{k} = {ct}\n")

        cipher.append(ct)

    print(f"Final ciphertext bytes: {cipher}")
    return cipher


if __name__ == "__main__":
    # Example parameters (small N=8 for clarity)
    initial_S   = list(range(8))          # Identity permutation [0,1,…7]
    key_stream  = [1,2,3,6,1,2,3,6]       # Example key repeated to length 8
    plaintext   = [1,2,2,2]               # Four data bytes to encrypt

    # 1) KSA: prepare S
    S = rc4_state(initial_S, key_stream)

    # 2) PRGA: generate keystream, encrypt plaintext
    rc4_prga(S, plaintext, len(plaintext))


=== RC4 Stream Cipher Demo ===

>> Initialization
   Starting S (identity): [0, 1, 2, 3, 4, 5, 6, 7]
   Expanded Key array K:  [1, 2, 3, 6, 1, 2, 3, 6]

>> Key-Scheduling Algorithm (KSA) Steps
  Step i= 0:
    j = (0 + S[0]=0 + K[0]=1) mod 8 → 1
    Swap S[0] (0) with S[1] (1)
    New S: [1, 0, 2, 3, 4, 5, 6, 7]

  Step i= 1:
    j = (1 + S[1]=0 + K[1]=2) mod 8 → 3
    Swap S[1] (0) with S[3] (3)
    New S: [1, 3, 2, 0, 4, 5, 6, 7]

  Step i= 2:
    j = (3 + S[2]=2 + K[2]=3) mod 8 → 0
    Swap S[2] (2) with S[0] (1)
    New S: [2, 3, 1, 0, 4, 5, 6, 7]

  Step i= 3:
    j = (0 + S[3]=0 + K[3]=6) mod 8 → 6
    Swap S[3] (0) with S[6] (6)
    New S: [2, 3, 1, 6, 4, 5, 0, 7]

  Step i= 4:
    j = (6 + S[4]=4 + K[4]=1) mod 8 → 3
    Swap S[4] (4) with S[3] (6)
    New S: [2, 3, 1, 4, 6, 5, 0, 7]

  Step i= 5:
    j = (3 + S[5]=5 + K[5]=2) mod 8 → 2
    Swap S[5] (5) with S[2] (1)
    New S: [2, 3, 5, 4, 6, 1, 0, 7]

  Step i= 6:
    j = (2 + S[6]=0 + K[6]=3) mod 8 → 5
    Swap S[6] (0) with